In [ ]:
# Proof of Concept: Multi-Agent System
#
# This notebook outlines the structure of a multi-agent system for the CanvasBot AI design platform.

from langchain.schema import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Union, Optional
import json


In [ ]:
# 1. Define Pydantic Models for Agentic Output

class CopywritingResult(BaseModel):
    """The result of the copywriting agent."""
    headline: str
    body: str

class DesignLayout(BaseModel):
    """A model to represent a print design layout."""
    width: int = Field(default=612)
    height: int = Field(default=792)
    elements: List[dict] = Field(..., description="The list of design elements on the page.")


In [ ]:
# 2. Initialize the LLM
# We'll use the same local vLLM server as in the poc.ipynb notebook.

llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="none",
    model="Qwen/Qwen2.5-3B-Instruct-AWQ",
    temperature=0.0
)


In [ ]:
# 3. Define the Agents

def copywriting_agent(prompt: str) -> CopywritingResult:
    """Generates marketing copy from a user prompt."""
    print("--- Running Copywriting Agent ---")

    system_prompt = f"""
You are a world-class marketing copywriter. Your job is to take a user's request and generate compelling copy for a print advertisement.

You must generate a headline and a body text.

Respond with ONLY a JSON object that conforms to the following Pydantic schema:
{json.dumps(CopywritingResult.model_json_schema(), indent=2)}
"""

    response = llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ])

    try:
        result = CopywritingResult(**json.loads(response.content))
        print("Copywriting Agent Result:")
        print(result)
        return result
    except (json.JSONDecodeError, TypeError) as e:
        print(f"Error parsing copywriting agent response: {e}")
        return None

def layout_agent(copy: CopywritingResult) -> DesignLayout:
    """Generates a design layout from the provided copy."""
    print("\n--- Running Layout Agent ---")

    system_prompt = f"""
You are a world-class graphic design layout agent. Your job is to take marketing copy and generate a structured JSON object that represents a print-ready design.

The JSON object must conform to the following JSON schema:
{json.dumps(DesignLayout.model_json_schema(), indent=2)}

- The page size is a standard letter size (width: 612, height: 792).
- Place the elements logically on the page. The origin (0,0) is at the bottom-left corner.
- Use the provided copy to create text elements.
- Do not include any image elements for now.
- Respond with ONLY the JSON object, without any additional text or explanations.
"""

    user_prompt = f"Create a layout for the following copy:\n\nHeadline: {copy.headline}\nBody: {copy.body}"

    response = llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ])

    try:
        result = DesignLayout(**json.loads(response.content))
        print("Layout Agent Result:")
        print(result.model_dump_json(indent=2))
        return result
    except (json.JSONDecodeError, TypeError) as e:
        print(f"Error parsing layout agent response: {e}")
        return None


In [ ]:
# 4. Orchestrate the Agents

def orchestrator(prompt: str):
    """
    Orchestrates the copywriting and layout agents to generate a design from a user prompt.
    """
    print("--- Starting Orchestration ---")

    # 1. Run the copywriting agent
    copy_result = copywriting_agent(prompt)

    if copy_result is None:
        print("Orchestration failed: Copywriting agent did not return a valid result.")
        return

    # 2. Run the layout agent
    layout_result = layout_agent(copy_result)

    if layout_result is None:
        print("Orchestration failed: Layout agent did not return a valid result.")
        return

    print("\n--- Orchestration Complete ---")
    # In a real application, this layout would be sent to the rendering engine.
    # For this PoC, we'll just print the final result.
    print("\nFinal Design Layout:")
    print(layout_result.model_dump_json(indent=2))


In [ ]:
# 5. Run the Orchestrator

user_prompt = "Make me a modern flyer for my coffee shop grand opening on Saturday, offering 20% off."
orchestrator(user_prompt)


In [ ]:
# Next Steps
#
# This notebook demonstrates a basic multi-agent system. The next steps would be to:
# 1. Implement a more robust orchestration framework like LangGraph.
# 2. Add an image generation agent.
# 3. Integrate the FastAPI rendering endpoint to generate a PDF from the final layout.
# 4. Add error handling and retry logic.
